# Monocyte Table Provenance

This notebook reproduces Supplementary Table `tab:monocyte_papers` from raw `markers.json` files.

It builds monocyte profiles with the same filtering logic used in the cross-paper analysis, attaches paper citation keys and brief study context, computes the monocyte consistency summary statistic, and writes provenance artifacts to `analysis/tables/`.


In [1]:
from __future__ import annotations

import json
import os
import re
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "biorxiv" / "meca").exists():
    ROOT = ROOT.parent

MECA_DIR = ROOT / "data" / "biorxiv" / "meca"
assert MECA_DIR.exists(), f"Could not locate MECA directory: {MECA_DIR}"

# Stable mapping for the 29 monocyte papers included in the table.
# key: meca paper_id folder
PAPER_METADATA = {
    "0a537c3c-6dba-1014-a3da-a09fc644eb0b": {"citation_key": "Preglej2025", "context": "Rheumatoid arthritis"},
    "1e0a6787-73c3-1014-991b-bc3d9c192616": {"citation_key": "Urbanski2025biorxiv640483", "context": "Sjogren's disease PBMC"},
    "1efd9fbe-6f02-1014-8df4-f37a8d806beb": {"citation_key": "Bachurski2025", "context": "EV immunomodulation assay"},
    "1f87890c-7e00-1014-91de-cf40554be0ec": {"citation_key": "Imoto2025", "context": "Method benchmark dataset"},
    "307b2bd7-6cb9-1014-a6b4-a58a7220bba3": {"citation_key": "Wang2020celda", "context": "Method paper (scRNA)"},
    "38f92858-6c77-1014-a70a-b9d4e7b85d53": {"citation_key": "Wu2022biorxiv520033", "context": "Azoospermia testis"},
    "39510275-6e7f-1014-81d5-da3b7edd0de0": {"citation_key": "Shangguan2021", "context": "HIV vaccine response"},
    "396ad855-6e2e-1014-85f5-9bd33dea0f81": {"citation_key": "Zhao2024", "context": "Type 2 diabetes PBMC"},
    "5200cce7-72d7-1014-98f6-994ae061fa4b": {"citation_key": "Obolenska2025", "context": "Placenta pregnancy"},
    "55d0fee5-7079-1014-8ef3-9aa3522a8f21": {"citation_key": "Rigby2023", "context": "Type I interferon response"},
    "876ca999-6c0e-1014-8ed9-e7563228589e": {"citation_key": "MariaRanzoni2020biorxiv080259", "context": "Developmental haematopoiesis"},
    "8a79edc5-6e0d-1014-bd87-aa1de4068967": {"citation_key": "Asaba2024", "context": "COVID-19 BAL fluid"},
    "9912495b-7074-1014-acf9-d95bb2127278": {"citation_key": "Gavriilidis2024", "context": "Severe COVID-19"},
    "9f9eded3-6d6e-1014-996d-af8f6f192d54": {"citation_key": "Lawlor2020", "context": "Stimulated PBMCs"},
    "ab05f819-6c04-1014-81aa-9646e853264a": {"citation_key": "Satpathy2019", "context": "Immune chromatin atlas"},
    "b3e65ef0-6e38-1014-8f8f-a6ec49a73092": {"citation_key": "Sherwood2024zika", "context": "Zika brain-tumor model"},
    "b4865554-70d4-1014-b98c-d54405b178ea": {"citation_key": "Shemonti2025", "context": "Flow cytometry method"},
    "bb90a40e-70a1-1014-887a-db6e99ca95cb": {"citation_key": "Xu2025pdac", "context": "Pancreatic cancer (PDAC)"},
    "c07140e5-7138-1014-aa16-8cb2f542cef3": {"citation_key": "Baldonado2025biorxiv689843", "context": "Atopic dermatitis"},
    "c245f005-7ab6-1014-adbd-b49bd8bfffbc": {"citation_key": "Alkhadrawi2025", "context": "Feature selection benchmark"},
    "c8264099-6fc3-1014-9d27-9ae05a0c1d33": {"citation_key": "Yu2024hcnetlas", "context": "Disease genetics atlas"},
    "d2042444-7138-1014-94ac-a1f590fd3173": {"citation_key": "Wang2025icc", "context": "Intrahepatic cholangiocarcinoma"},
    "d249a77b-6c97-1014-8df4-e171a53e6bf0": {"citation_key": "Zhao2022biorxiv494592", "context": "Checkpoint inhibitor toxicity"},
    "dbf486c9-6c54-1014-9bc5-bf4d8f303013": {"citation_key": "Li2021aml", "context": "Acute myeloid leukemia"},
    "de6d33dd-734f-1014-a302-e21bedf0ba65": {"citation_key": "Luo2025biorxiv676755", "context": "CAR-T solid tumors"},
    "e6e04172-6c15-1014-9229-c57ac33d21f6": {"citation_key": "Yang2019decontx", "context": "Ambient RNA correction"},
    "ebaf49e5-7438-1014-8951-c3e925f71d14": {"citation_key": "Jeong2025biorxiv632256", "context": "ML method benchmark"},
    "f3be1d12-72db-1014-a9a0-ac8d78e3151e": {"citation_key": "Cheng2024glio", "context": "Glioblastoma microenvironment"},
    "f787a6d9-6ce3-1014-a3f6-a662a1333571": {"citation_key": "Leach2020biorxiv070839", "context": "Lung and lymph node"},
}

MIN_MARKERS = 3


In [2]:
# Build (paper_id, cell_type) -> set of marker IDs
profiles: dict[tuple[str, str], set[str]] = {}
gene_id_to_name: dict[str, str] = {}

for folder in sorted(os.listdir(MECA_DIR)):
    markers_path = MECA_DIR / folder / "markers.json"
    if not markers_path.exists():
        continue

    markers = json.loads(markers_path.read_text())
    for rec in markers:
        if str(rec.get("organism", "")).strip().lower() != "homo_sapiens":
            continue

        cell_type = str(rec.get("group_name", "")).strip().upper()
        gene_id = rec.get("feature_id")
        gene_name = str(rec.get("feature_name", "")).strip().upper()
        if not cell_type or not gene_id or not gene_name:
            continue

        key = (folder, cell_type)
        profiles.setdefault(key, set()).add(gene_id)
        gene_id_to_name[gene_id] = gene_name

filtered = {k: v for k, v in profiles.items() if len(v) >= MIN_MARKERS}
monocyte_profiles = {(pid, ct): genes for (pid, ct), genes in filtered.items() if "MONOCYTE" in ct}

# Deduplicate by paper: keep the cell type with the most extracted markers.
paper_to_best: dict[str, tuple[str, set[str]]] = {}
for (paper_id, cell_type), genes in monocyte_profiles.items():
    if paper_id not in paper_to_best or len(genes) > len(paper_to_best[paper_id][1]):
        paper_to_best[paper_id] = (cell_type, genes)

# Compute mean pairwise Jaccard among selected paper-level profiles.
paper_ids = sorted(paper_to_best)
jaccards = []
for a, b in combinations(paper_ids, 2):
    ga = paper_to_best[a][1]
    gb = paper_to_best[b][1]
    jaccards.append(len(ga & gb) / len(ga | gb))

mean_jaccard = float(np.mean(jaccards)) if jaccards else 0.0
median_jaccard = float(np.median(jaccards)) if jaccards else 0.0

rows = []
for paper_id in paper_ids:
    cell_type, genes = paper_to_best[paper_id]
    genes_sorted = sorted(gene_id_to_name[g].upper() for g in genes)
    if paper_id not in PAPER_METADATA:
        raise KeyError(f"Missing metadata mapping for paper_id={paper_id}")
    meta = PAPER_METADATA[paper_id]
    rows.append({
        "paper_id": paper_id,
        "citation_key": meta["citation_key"],
        "cell_type_label": cell_type,
        "context": meta["context"],
        "n_markers": len(genes_sorted),
        "markers": ", ".join(genes_sorted),
        "markers_list": genes_sorted,
    })

provenance_df = pd.DataFrame(rows)

assert len(provenance_df) == 29, f"Expected 29 papers, found {len(provenance_df)}"
print(f"Monocyte profiles before paper-level dedup: {len(monocyte_profiles)}")
print(f"Monocyte papers after dedup: {len(provenance_df)}")
print(f"Mean pairwise Jaccard: {mean_jaccard:.3f}")
print(f"Median pairwise Jaccard: {median_jaccard:.3f}")
provenance_df[["citation_key", "cell_type_label", "context", "n_markers"]].head(10)


Monocyte profiles before paper-level dedup: 38
Monocyte papers after dedup: 29
Mean pairwise Jaccard: 0.063
Median pairwise Jaccard: 0.000


,citation_key,cell_type_label,context,n_markers
0,Preglej2025,MONOCYTE,Rheumatoid arthritis,6
1,Urbanski2025biorxiv640483,ISG+ AP1LOW CLASSICAL MONOCYTES,Sjogren's disease PBMC,5
2,Bachurski2025,MONOCYTE,EV immunomodulation assay,6
3,Imoto2025,CD14+ MONOCYTE,Method benchmark dataset,4
4,Wang2020celda,CD14+ MONOCYTE,Method paper (scRNA),5
5,Wu2022biorxiv520033,MONOCYTES/MACROPHAGES,Azoospermia testis,3
6,Shangguan2021,MONOCYTE,HIV vaccine response,5
7,Zhao2024,CD14+ MONOCYTE,Type 2 diabetes PBMC,8
8,Obolenska2025,MONOCYTE,Placenta pregnancy,3
9,Rigby2023,CLASSICAL MONOCYTE,Type I interferon response,10


In [3]:
# Write provenance artifacts
out_dir = ROOT / "analysis" / "tables"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "monocyte_different_marker_genes_provenance.csv"
json_path = out_dir / "monocyte_different_marker_genes_provenance.json"
tex_rows_path = out_dir / "monocyte_different_marker_genes_rows.tex"

provenance_df.drop(columns=["markers_list"]).to_csv(csv_path, index=False)

payload = {
    "summary": {
        "min_markers": MIN_MARKERS,
        "profiles_before_dedup": len(monocyte_profiles),
        "papers_after_dedup": len(provenance_df),
        "mean_pairwise_jaccard": round(mean_jaccard, 3),
        "median_pairwise_jaccard": round(median_jaccard, 3),
    },
    "rows": provenance_df.to_dict(orient="records"),
}
json_path.write_text(json.dumps(payload, indent=2))

BS = chr(92)
latex_rows = []
for _, r in provenance_df.iterrows():
    cell_type = r["cell_type_label"].replace("MONOCYTES/MACROPHAGES", f"MONOCYTES/{BS}allowbreak MACROPHAGES")
    latex_rows.append(
        f"{BS}citep{{{r['citation_key']}}} & {cell_type} & {r['context']} & {r['markers']} {BS}{BS} [4pt]"
    )
tex_rows_path.write_text("\n".join(latex_rows) + "\n")

print(f"Wrote: {csv_path.relative_to(ROOT)}")
print(f"Wrote: {json_path.relative_to(ROOT)}")
print(f"Wrote: {tex_rows_path.relative_to(ROOT)}")

# Optional consistency check against the current manuscript table.
main_tex = (ROOT / "paper" / "main.tex").read_text()
start = main_tex.index(f"{BS}label{{tab:monocyte_papers}}")
end = main_tex.index(f"{BS}bottomrule", start)
block = main_tex[start:end]

table_rows = []
for line in block.splitlines():
    line = line.strip()
    if not line.startswith(f"{BS}citep{{"):
        continue
    line = line.split(f"{BS}{BS}")[0]
    parts = [p.strip() for p in line.split("&")]
    if len(parts) != 4:
        continue
    cite_key = parts[0].split("{", 1)[1].rsplit("}", 1)[0]
    cell_type = parts[1].upper().replace(f"{BS}ALLOWBREAK", "")
    cell_type = re.sub(r"\s*/\s*", "/", cell_type)
    context = parts[2]
    markers = ", ".join(sorted(g.strip().upper() for g in parts[3].split(",") if g.strip()))
    table_rows.append((cite_key, cell_type, context, markers))

prov_rows = []
for _, r in provenance_df.iterrows():
    ct = r["cell_type_label"].upper()
    ct = re.sub(r"\s*/\s*", "/", ct)
    markers = ", ".join(sorted(g.upper() for g in r["markers_list"]))
    prov_rows.append((r["citation_key"], ct, r["context"], markers))

if Counter(table_rows) != Counter(prov_rows):
    diff_table = Counter(table_rows) - Counter(prov_rows)
    diff_prov = Counter(prov_rows) - Counter(table_rows)
    raise AssertionError(
        f"Manuscript table mismatch. table-only={sum(diff_table.values())}, provenance-only={sum(diff_prov.values())}"
    )

print("Manuscript monocyte table matches generated provenance rows.")


Wrote: analysis/tables/monocyte_different_marker_genes_provenance.csv
Wrote: analysis/tables/monocyte_different_marker_genes_provenance.json
Wrote: analysis/tables/monocyte_different_marker_genes_rows.tex
Manuscript monocyte table matches generated provenance rows.
